# 01 — Muestra de revisión cualitativa

Este notebook genera una muestra reducida para revisar cualitativamente el comportamiento del modelo de sentimiento. No se utiliza para entrenar modelos ni para calcular métricas supervisadas.

In [7]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_INPUT = PROJECT_ROOT / "data" / "input"
DATA_LABELS = PROJECT_ROOT / "data" / "labels"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_LABELS.mkdir(parents=True, exist_ok=True)

TWEETS_INPUT = DATA_INPUT / "tweets_limpios_2020_2025_minecogob.csv"
TWEETS_CLASIFICADOS = DATA_PROCESSED / "tweets_clasificados_sentimiento.csv"
SALIDA = DATA_LABELS / "muestra_revision_cualitativa.csv"

print("Proyecto:", PROJECT_ROOT)
print("Entrada tweets:", TWEETS_INPUT)

Proyecto: C:\Users\NADIA\TFG\src\analisis_del_dato\analisis_del_dato
Entrada tweets: C:\Users\NADIA\TFG\src\analisis_del_dato\analisis_del_dato\data\input\tweets_limpios_2020_2025_minecogob.csv


## Carga de datos

Si ya existe el archivo de tweets clasificados, la muestra incluirá la etiqueta y confianza del modelo. Si todavía no existe, se genera la muestra con esas columnas vacías.

In [8]:
if TWEETS_CLASIFICADOS.exists():
    df = pd.read_csv(TWEETS_CLASIFICADOS)
    print("Se usará el corpus ya clasificado por el modelo.")
else:
    df = pd.read_csv(TWEETS_INPUT)
    df["sentimiento_modelo"] = ""
    df["confianza_modelo"] = np.nan
    print("Todavía no existe corpus clasificado; se genera muestra sin etiqueta de modelo.")

for c in ["fecha_dt", "mes"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

columnas_necesarias = ["id", "fecha_dt", "mes", "username", "contenido", "texto_limpio"]
faltantes = [c for c in columnas_necesarias if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas necesarias: {faltantes}")

print(df.shape)
display(df.head())

Se usará el corpus ya clasificado por el modelo.
(4590, 21)


,id,fecha_dt,mes,username,contenido,texto_limpio,sentimiento_modelo,confianza_modelo,prob_pos,prob_neg,...,fecha,hashtags,tipo,url_media,comentarios,retweets,likes,views,fecha_limpia,texto_modelo
0,2006087164900360491,2025-12-30 19:37:00,2025-12-01,@_minecogob,El ministro @carlos_cuerpo es entrevistado en ...,el ministro es entrevistado en el programa 'un...,neutro,0.768274,0.103679,0.128047,...,"Dec 30, 2025 · 7:37 PM UTC",NaN,Imagen,https://nitter.net/pic/orig/media%2FG9cNpJ6XAA...,0,1,3,528,"Dec 30, 2025 7:37 PM",el ministro es entrevistado en el programa 'un...
1,2005913464938614981,2025-12-30 08:06:00,2025-12-01,@_minecogob,"La inflación baja una décima en diciembre, al ...","la inflación baja una décima en diciembre, al ...",neutro,0.683655,0.099481,0.216864,...,"Dec 30, 2025 · 8:06 AM UTC",NaN,Imagen,https://nitter.net/pic/orig/media%2FG9ZvqkDXIA...,0,2,2,367,"Dec 30, 2025 8:06 AM","la inflación baja una décima en diciembre, al ..."
2,2003440348534833308,2025-12-23 12:19:00,2025-12-01,@_minecogob,"El @es_INE confirma que el PIB creció un 0,6% ...","el confirma que el pib creció un 0,6% en 3t, c...",neutro,0.586499,0.325938,0.087562,...,"Dec 23, 2025 · 12:19 PM UTC",NaN,Imagen,https://nitter.net/pic/orig/media%2FG82mYDZXAA...,0,3,2,1445,"Dec 23, 2025 12:19 PM","el confirma que el pib creció un 0,6% en 3t, c..."
3,2003361375310238130,2025-12-23 07:05:00,2025-12-01,@_minecogob,El ministro @carlos_cuerpo es entrevistado en ...,el ministro es entrevistado en . ⏰a partir de ...,neutro,0.872291,0.050628,0.077081,...,"Dec 23, 2025 · 7:05 AM UTC",NaN,Imagen,https://nitter.net/pic/orig/media%2FG81ejPFXYA...,2,3,5,635,"Dec 23, 2025 7:05 AM",el ministro es entrevistado en . ⏰a partir de ...
4,2003037679471022544,2025-12-22 09:39:00,2025-12-01,@_minecogob,📈Máximo histórico de exportaciones españolas d...,📈máximo histórico de exportaciones españolas d...,neutro,0.672476,0.154434,0.173091,...,"Dec 22, 2025 · 9:39 AM UTC",NaN,Imagen,https://nitter.net/pic/orig/media%2FG8w4JrJXoA...,0,6,6,1383,"Dec 22, 2025 9:39 AM",📈máximo histórico de exportaciones españolas d...


## Muestreo: 60 tweets en total

Se seleccionan 10 tweets por año entre 2020 y 2025. La revisión manual debe marcar únicamente si la clasificación del modelo parece `correcta`, `dudosa` o `incorrecta`. Esta revisión es cualitativa y no se usa para entrenar.

In [9]:
df_valido = df.dropna(subset=["texto_limpio", "fecha_dt"]).copy()
df_valido = df_valido[df_valido["texto_limpio"].astype(str).str.strip() != ""].copy()
df_valido["year"] = df_valido["fecha_dt"].dt.year

partes = []
for year in range(2020, 2026):
    sub = df_valido[df_valido["year"] == year]
    n = min(10, len(sub))
    if n > 0:
        partes.append(sub.sample(n=n, random_state=42))

muestra = pd.concat(partes).sample(frac=1, random_state=42).reset_index(drop=True)

for col in ["sentimiento_modelo", "confianza_modelo"]:
    if col not in muestra.columns:
        muestra[col] = "" if col == "sentimiento_modelo" else np.nan

columnas = ["id", "fecha_dt", "mes", "username", "contenido", "texto_limpio", "sentimiento_modelo", "confianza_modelo"]
muestra_export = muestra[columnas].copy()
muestra_export["revision_manual"] = ""
muestra_export["notas"] = ""

muestra_export.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print("Muestra exportada en:", SALIDA)
print("Número de tweets:", len(muestra_export))
print("Distribución por año:")
print(muestra["year"].value_counts().sort_index())
display(muestra_export.head())

Muestra exportada en: C:\Users\NADIA\TFG\src\analisis_del_dato\analisis_del_dato\data\labels\muestra_revision_cualitativa.csv
Número de tweets: 60
Distribución por año:
year
2020    10
2021    10
2022    10
2023    10
2024    10
2025    10
Name: count, dtype: int64


,id,fecha_dt,mes,username,contenido,texto_limpio,sentimiento_modelo,confianza_modelo,revision_manual,notas
0,1324305478919327744,2020-11-05 11:00:00,2020-11-01,@_minecogob,La VP @NadiaCalvino en @EspejoPublico:\n\nTene...,la vp en : tenemos una buena base para la recu...,neutro,0.597409,,
1,1316270261478600705,2020-10-14 06:51:00,2020-10-01,@_minecogob,🏛 La vicepresidenta @NadiaCalvino interviene e...,🏛 la vicepresidenta interviene en la sesión de...,neutro,0.690117,,
2,1640978377917046785,2023-03-29 07:25:00,2023-03-01,@_minecogob,🏛️@NadiaCalvino: Nuestra política económica es...,🏛️ : nuestra política económica es la correcta...,neutro,0.636139,,
3,1758165190959366552,2024-02-15 16:23:00,2024-02-01,@_minecogob,El crecimiento económico estará acompañado de ...,el crecimiento económico estará acompañado de ...,neutro,0.750497,,
4,1407324756991528961,2021-06-22 13:09:00,2021-06-01,@_minecogob,La VP @NadiaCalvino en @Congreso_Es:\n\nLas re...,la vp en : las reformas e inversiones del plan...,neutro,0.571444,,


## Guía de revisión cualitativa

- `correcta`: la etiqueta del modelo refleja razonablemente el tono del tweet.
- `dudosa`: el tweet podría interpretarse de más de una forma o el lenguaje institucional es ambiguo.
- `incorrecta`: la etiqueta del modelo no refleja el tono del tweet.

Esta revisión no sustituye una evaluación supervisada. Su objetivo es documentar limitaciones y errores típicos del modelo preentrenado aplicado a discurso institucional.